# Step 1: Dataset Discovery
Automatically inspect products, orders, prior orders, aisles, and departments.

In [1]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append('..')
from config import *

def analyze_df(name, path):
    print(f"--- Analyzing {name} ---")
    df = pd.read_csv(path)
    print(f"Shape: {df.shape}")
    print("\nData Types:\n", df.dtypes)
    print("\nMissing Values:\n", df.isnull().sum())
    print("\nDuplicate Rows:", df.duplicated().sum())
    print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print("\nSample:\n", df.head())
    print("\n" + "="*50 + "\n")
    return df


In [2]:
# 1. Departments
dept_df = analyze_df("Departments", RAW_DEPARTMENTS)

--- Analyzing Departments ---
Shape: (21, 2)

Data Types:
 department_id     int64
department       object
dtype: object

Missing Values:
 department_id    0
department       0
dtype: int64

Duplicate Rows: 0

Memory Usage: 0.00 MB

Sample:
    department_id department
0              1     frozen
1              2      other
2              3     bakery
3              4    produce
4              5    alcohol




In [3]:
# 2. Aisles
aisles_df = analyze_df("Aisles", RAW_AISLES)

--- Analyzing Aisles ---
Shape: (134, 2)

Data Types:
 aisle_id     int64
aisle       object
dtype: object

Missing Values:
 aisle_id    0
aisle       0
dtype: int64

Duplicate Rows: 0

Memory Usage: 0.01 MB

Sample:
    aisle_id                       aisle
0         1       prepared soups salads
1         2           specialty cheeses
2         3         energy granola bars
3         4               instant foods
4         5  marinades meat preparation




In [4]:
# 3. Products
products_df = analyze_df("Products", RAW_PRODUCTS)

--- Analyzing Products ---
Shape: (49688, 4)

Data Types:
 product_id        int64
product_name     object
aisle_id          int64
department_id     int64
dtype: object

Missing Values:
 product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

Duplicate Rows: 0

Memory Usage: 4.94 MB

Sample:
    product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4           5                          Green Chile Anytime Sauce         5   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13  




In [5]:
# 4. Orders
orders_df = analyze_df("Orders", RAW_ORDERS)

--- Analyzing Orders ---


Shape: (3421083, 7)

Data Types:
 order_id                    int64
user_id                     int64
eval_set                   object
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

Missing Values:
 order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64



Duplicate Rows: 0

Memory Usage: 332.71 MB

Sample:
    order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  




In [6]:
# 5. Order Products (Prior) - NOTE: This is a large file.
# Using chunking or specifying dtypes might be needed later, but for discovery we will load a sample if it's too big, or just load it entirely if we have enough RAM.
try:
    prior_df = analyze_df("Order Products Prior", RAW_ORDER_PRODUCTS_PRIOR)
except MemoryError:
    print("Memory Error: Loading only a chunk")
    prior_df = pd.read_csv(RAW_ORDER_PRODUCTS_PRIOR, nrows=100000)
    print(f"Loaded a sample of 100,000 rows. Memory Usage: {prior_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


--- Analyzing Order Products Prior ---


Shape: (32434489, 4)

Data Types:
 order_id             int64
product_id           int64
add_to_cart_order    int64
reordered            int64
dtype: object



Missing Values:
 order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64



Duplicate Rows: 0

Memory Usage: 989.82 MB

Sample:
    order_id  product_id  add_to_cart_order  reordered
0         2       33120                  1          1
1         2       28985                  2          1
2         2        9327                  3          0
3         2       45918                  4          1
4         2       30035                  5          0




## Data Dictionary & ER Diagram Explanation
- **departments.csv**: `department_id` (PK), `department` (Name)
- **aisles.csv**: `aisle_id` (PK), `aisle` (Name)
- **products.csv**: `product_id` (PK), `product_name`, `aisle_id` (FK), `department_id` (FK)
- **orders.csv**: `order_id` (PK), `user_id`, `eval_set`, `order_number`, `order_dow`, `order_hour_of_day`, `days_since_prior_order`
- **order_products__prior.csv**: Composite PK (`order_id`, `product_id`), `add_to_cart_order`, `reordered`
